<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_Duck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libaries

In [84]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb

In [85]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [86]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [87]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tbl7OYOXduME11uh7")   # Slut-targets

In [88]:
# Explode and rename
t0 = tabel0.copy().explode("Targets (policy targets)").rename(columns={"Targets (policy targets)": "target_id"})
t1 = tabel1.copy().explode("Target Group").rename(columns={"Target Group": "target_group_id"})
t2 = tabel2.copy().explode("Targets").rename(columns={"Targets": "target_id"})

In [89]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3", tabel3)

In [90]:
# SQL query
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."Target Group"        AS target_group,
    t3."Target name"         AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3 t3 ON t2.target_id = t3.id
"""

results = duckdb.sql(query).df()

In [91]:
# Trin 1: forkort og saml labels
results['final_target'] = results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels = results['policy_source']
middle_labels = results['target_group']
target_labels = results['final_target']
all_labels = pd.concat([source_labels, middle_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [92]:
# Trin 2: forbindelser
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index),
    'target': middle_labels.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})
all_links = pd.concat([links1, links2])

In [93]:
# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color="lightgray"
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=2000)
fig.show()

#Selektering grøn trepart

In [99]:
# prompt: make a sankey diagram with only Aftale om et grønt Danmark (grøn trepart) as policy source

import pandas as pd
# Selektering grøn trepart
filtered_results = results[results['policy_source'] == 'Aftale om et grønt Danmark (grøn trepart)'].copy()

# Trin 1: forkort og saml labels
filtered_results['final_target'] = filtered_results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels_filtered = filtered_results['policy_source']
middle_labels_filtered = filtered_results['target_group']
target_labels_filtered = filtered_results['final_target']

# Only include labels that are present in the filtered data
all_labels_filtered = pd.concat([source_labels_filtered, middle_labels_filtered, target_labels_filtered])
unique_labels_filtered = pd.unique(all_labels_filtered)
label_to_index_filtered = {label: i for i, label in enumerate(unique_labels_filtered)}

# Trin 2: forbindelser for filtreret data
links1_filtered = pd.DataFrame({
    'source': source_labels_filtered.map(label_to_index_filtered),
    'target': middle_labels_filtered.map(label_to_index_filtered),
    'value': 1
})

links2_filtered = pd.DataFrame({
    'source': middle_labels_filtered.map(label_to_index_filtered),
    'target': target_labels_filtered.map(label_to_index_filtered),
    'value': 1
})

all_links_filtered = pd.concat([links1_filtered, links2_filtered])

# Remove any links where the source or target index is NaN (due to filtering)
all_links_filtered.dropna(subset=['source', 'target'], inplace=True)
all_links_filtered['source'] = all_links_filtered['source'].astype(int)
all_links_filtered['target'] = all_links_filtered['target'].astype(int)


# Trin 3: Sankey-diagram for filtreret data
fig_filtered = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels_filtered),
    ),
    link=dict(
        source=all_links_filtered['source'],
        target=all_links_filtered['target'],
        value=all_links_filtered['value'],
        color="lightgray"
    )
)])

fig_filtered.update_layout(title_text="Aftale om et grønt Danmark (grøn trepart) → Target Group → Target", font_size=12, height=2500)
fig_filtered.show()

In [101]:
# prompt: download an interactive html file

fig.write_html("sankey_diagram.html")
from google.colab import files
files.download("sankey_diagram.html")

fig_filtered.write_html("sankey_diagram_filtered.html")
files.download("sankey_diagram_filtered.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>